# P4 — Files, libraries and the scientific stack

**Python primer · session 4 of 5**

## What this session is for

Everything so far lived inside the notebook. Real work does not. Data arrives
in files and the code that handles it is mostly written by other people.

So this session is about the two things that connect your code to the world.
**Files**: text, CSV, JSON and HDF5, the format that scientific data usually
comes in. And **libraries**: how to import them, where they come from and the
four that carry this course.

It ends with pictures. You write a week of temperatures to a file, read it
back and plot it. You draw a tone and the same tone under a hiss. Then you
draw a straight line through forty years of yearly temperatures. Every number
in this session is made up for the exercise.

## Text files

A text file is lines of characters and nothing else. Use it for a log, a list of
names or a note to your future self.

The oldest and simplest case. Open a file, read it, close it. A class register
is a text file with one name on each line.

In [ ]:
names = ["Anna", "Bruno", "Chiara", "Dario", "Elisa"]

with open('register.txt', 'w') as f:
    for name in names:
        f.write(name + '\n')

print('written')

The `with` block is the important part. It opens the file, gives it to you as
`f` and closes it when the block ends, **even if something inside fails**.
Files opened without `with` stay open when an error interrupts you. On a
shared machine that eventually becomes somebody else's problem.

The mode is the second argument:

| Mode | Meaning |
|---|---|
| `'r'` | read, the default |
| `'w'` | write, **and it deletes what was there** |
| `'a'` | append, add to the end |

Three ways to read it back:

In [ ]:
with open('register.txt', 'r') as f:
    whole = f.read()               # everything, as one string
print(repr(whole))

with open('register.txt', 'r') as f:
    lines = f.readlines()          # a list of lines
print(lines)

with open('register.txt', 'r') as f:
    for line in f:                 # one line at a time -- best for big files
        print(line.strip())

`line.strip()` removes the newline character at the end. Forgetting it is the
reason your printed output has blank lines in it.

The third form reads one line at a time and never holds the whole file in
memory. For a register of a few names it makes no difference. For a data
file of several gigabytes it is the difference between working and not.

### When the file is not there

In [ ]:
try:
    with open('does_not_exist.txt', 'r') as f:
        print(f.read())
except FileNotFoundError:
    print("not there -- check the path and where you are")

import os
print("we are in:", os.getcwd())

Most "the file is not there" errors are really "I am not where I think I am".
`os.getcwd()` tells you the working directory and `os.listdir()` tells you what
is in it. Check those two before doubting the file.

## CSV: rows and columns

A CSV file is a table as plain text, with one row on each line and commas
between the columns. Use it for any table that another program or a
spreadsheet must read.

**CSV** means comma separated values. We write one: the marks of six students
in three exams. The marks are made up. Each row is a student and each row is a
tuple of a name and three marks.

In [ ]:
import csv

rows = [("Anna", 27, 30, 24), ("Bruno", 21, 18, 25),
        ("Chiara", 30, 28, 29), ("Dario", 18, 22, 19),
        ("Elisa", 25, 27, 30), ("Franco", 19, 24, 21)]

with open('marks.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['name', 'maths', 'physics', 'chemistry'])
    for row in rows:
        writer.writerow(row)

`csv.writer` puts the commas in for you and `newline=''` stops a blank line
appearing between rows on some systems. Look at what is in the file:

In [ ]:
path = 'marks.csv'

with open(path, 'r') as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i == 3:
            break

The first line is the **header**, naming the columns. The `csv` module in the
standard library reads it properly. It handles quoted fields and commas inside
values. Naive splitting on commas does not:

In [ ]:
with open(path, 'r') as f:
    reader = csv.DictReader(f)          # each row becomes a dictionary
    table_rows = list(reader)

print(len(table_rows), "students")
first = table_rows[0]
print(first['name'], first['maths'])

`DictReader` gives each row as a dictionary keyed by the column names, which
is far easier to read than `row[1]`.

Note that everything is a **string**. CSV has no types. Converting is your
job. Here is the mean mark of each student:

In [ ]:
with open(path, 'r') as f:
    for row in csv.DictReader(f):
        three = [int(row['maths']), int(row['physics']),
                 int(row['chemistry'])]
        print(f"{row['name']:8s} {sum(three) / len(three):.2f}")

### The same file with pandas

For tables, the `pandas` library does the whole job in one line and gives you
something you can compute with.

In [ ]:
import pandas as pd

table = pd.read_csv(path)
print(table.shape)                       # rows, columns
table.head(3)

In [ ]:
exams = ['maths', 'physics', 'chemistry']
table['mean'] = table[exams].mean(axis=1)    # one mean per row

print(table[['name', 'mean']].round(2))
print(f"class mean: {table['mean'].mean():.2f}")

low = table[table['mean'] < 22]
print(low['name'].tolist())

`axis=1` means across the columns of each row. `table['mean'] < 22` is the
boolean mask of last session and `table[mask]` keeps the rows where it is
`True`. That is the core of pandas: read a table, take a column, select rows.
The next cells show what the table gives you for free.

A **DataFrame** is a table with named columns and a row index. `table` is one.
Its `describe` method gives, for each column, the count, the mean, the
spread, the minimum, the quartiles and the maximum. The spread is the standard
deviation. The quartiles cut the sorted values into four equal parts.

In [ ]:
table[exams].describe().round(2)

`sort_values` puts the rows in order of one column. Here the best students
come first.

In [ ]:
ranked = table.sort_values('mean', ascending=False)
print(ranked[['name', 'mean']].round(2))

One column has its own `mean`. Compare the mean of one exam with the class
mean of all the marks. Then take the mean of every exam at once.

In [ ]:
print(f"maths {table['maths'].mean():.2f}")
print(f"class {table['mean'].mean():.2f}")
print(table[exams].mean().round(2))

A DataFrame draws itself. `plot.bar` makes one group of bars for each row
and `set_index('name')` puts the names on the horizontal axis. The call gives
back the axes, so you can label them. `pandas` draws with `matplotlib` and
`plt.show()` shows the figure.

In [ ]:
import matplotlib.pyplot as plt

by_name = table.set_index('name')[exams]
ax = by_name.plot.bar(figsize=(8, 3.5))
ax.set_xlabel('student')
ax.set_ylabel('mark')
ax.set_title('Three exams for six students (made up)')
plt.show()

A **histogram** counts how many values fall in each bin, which is a range of
values. `stack` puts the three exam columns into one long column of 18 marks.

In [ ]:
marks = table[exams].stack()
print(len(marks), "marks")
ax = marks.hist(bins=6, figsize=(6, 3))
ax.set_xlabel('mark')
ax.set_ylabel('number of marks')
ax.set_title('All the marks of the class (made up)')
plt.show()

A boolean selection works on any column. Which students have a mean mark of
at least 24?

In [ ]:
good = table[table['mean'] >= 24]
print(good['name'].tolist())
print(len(good), "of", len(table))

> **Where to read more**
>
> The pandas documentation has two places to start: "pandas: Getting started"
> and the "User Guide". Both are at https://pandas.pydata.org/docs/ and the
> link is correct as of October 2026.

## JSON: nested data

JSON is text that holds a nested structure of names, lists and values. Use it
for a record, a configuration or anything a web service hands you.

**JSON** means JavaScript Object Notation. Python reads it as dictionaries and
lists, nested inside each other.

In [ ]:
import json

record = {
    "name": "Anna",
    "year": 2,
    "courses": ["maths", "physics"],
    "marks": {"maths": 27, "physics": 30},
}

This is the record of last session. Write it out, then read it back and dig
into the nesting.

In [ ]:
with open('student.json', 'w') as f:
    json.dump(record, f, indent=2)

with open('student.json', 'r') as f:
    back = json.load(f)

print(back["name"], back["courses"])
print(back["marks"]["physics"], "in physics")
print(back["marks"]["maths"], "in maths")

JSON maps onto Python containers exactly: an object becomes a dictionary, an
array becomes a list. That is why it is everywhere.

## Modules and libraries

A **module** is a Python file you can import. A **library** is a collection of
them, written by somebody else and installed on your machine.

In [ ]:
import math
print(math.sqrt(16), math.pi)

from math import sqrt, pi          # bring specific names in
print(sqrt(16), pi)

import numpy as np                 # give it a shorter name
print(np.sqrt(16))

Three forms, all used. `import x` keeps the name in front, which is clearest.
`from x import y` is convenient for a name used constantly. `import x as y` is
for the conventional short names, `np`, `pd` and `plt`. Everybody writes
them the same way and you should too.

Avoid `from x import *`. It dumps every name into your notebook and you can no
longer tell where anything came from.

**Where libraries come from.** They are installed with `pip`, from a public
index of hundreds of thousands of packages:

```bash
pip install h5py                # install
pip list                        # what is installed
```

You will not need to do this often here. The environment for the course
already has what the notebooks use. What matters is knowing that `import h5py`
works because somebody installed `h5py`. An `ImportError` means it is
missing rather than broken.

The four that carry this course and what each is for:

| Library | What it does |
|---|---|
| `numpy` | arrays and the arithmetic on them |
| `matplotlib` | plots |
| `scipy` | statistics, a straight line through points and much else |
| `pandas` | tables |

One more is used in this session: `h5py`, for HDF5 files.

## HDF5: how scientific data is stored

An HDF5 file is a binary file that holds arrays with their attributes, in
groups like folders. Use it when the data is large and numeric and must carry
its own description.

A text file with a million numbers in it would be slow and enormous. One
reading every second for a month is two and a half million. Scientific data is
stored in binary and the standard format here is **HDF5**.

An HDF5 file is a small file system inside one file:

- **groups** are like folders,
- **datasets** are arrays,
- **attributes** are small pieces of metadata attached to either.

Make one, to see the structure from the inside. It holds a week of hourly
temperatures, with the number of readings per hour stored beside it. The
temperatures are made up: a smooth rise and fall every 24 hours, which
`np.sin` gives, plus a small random scatter.

In [ ]:
import h5py

per_hour = 1                                   # readings per hour
hours = np.arange(7 * 24 * per_hour) / per_hour    # hour of the week
rng = np.random.default_rng(4)
daily = 18 + 6 * np.sin(2 * np.pi * (hours - 9) / 24)
temps = daily + rng.normal(0, 0.8, hours.size)

with h5py.File('example.h5', 'w') as f:
    g = f.create_group('week')
    d = g.create_dataset('temperatures', data=temps)
    d.attrs['units'] = 'degrees Celsius'
    d.attrs['readings_per_hour'] = per_hour
print('written')

Now open it again and look at what is inside.

In [ ]:
with h5py.File('example.h5', 'r') as f:
    print(list(f.keys()))
    print(list(f['week'].keys()))
    data = f['week/temperatures'][:]
    print(data.shape)
    for key, value in f['week/temperatures'].attrs.items():
        print(key, value)

> **Important — The colon that reads the data**
>
> `f['week/temperatures']` gives you a *reference* to the dataset, not the
> numbers. `f['week/temperatures'][:]` reads them into memory as an array.
>
> This is deliberate and it is why HDF5 handles files larger than your memory.
> You can look at the shape and the attributes of a hundred gigabyte dataset
> and then read only the slice you want.

### Finding your way in a file you did not write

`visititems` walks the whole tree and calls a function on every group and
dataset. It is the quickest way to see what a strange file holds.

In [ ]:
def show(name, obj):
    print(f"{name:22} {getattr(obj, 'shape', '')}")

with h5py.File('example.h5', 'r') as f:
    f.visititems(show)

Read the array and the metadata that gives it a time axis:

In [ ]:
with h5py.File('example.h5', 'r') as f:
    temps = f['week/temperatures'][:]
    per_hour = f['week/temperatures'].attrs['readings_per_hour']

hours = np.arange(temps.size) / per_hour      # hour of each reading
print(f"values     {temps.size}")
print(f"per hour   {per_hour}")
print(f"duration   {temps.size / per_hour / 24:.1f} days")
print(f"mean       {temps.mean():.1f} degrees")

Everything here is a habit worth copying. The number of readings per hour
came from the file rather than from memory. The time axis was built from it
rather than assumed.

Data you did not write always deserves that suspicion.

## Plotting it

`matplotlib` draws it. The pattern is always the same: make a figure and axes,
draw on the axes, label them.

In [ ]:
import matplotlib.pyplot as plt

days = hours / 24                # the time axis, in days

Make the figure and the axes, draw, label. Always in that order.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(days, temps)
ax.set_xlabel('time (days)')
ax.set_ylabel('temperature (degrees Celsius)')
ax.set_title('A week of hourly temperatures (made up)')
plt.show()

Check it the way P1 taught. A week is seven days and each day has one warm
afternoon. Count the peaks on the plot: seven. The file, the code and the
plot agree.

`pandas` draws the same week in one line. A `Series` is one column with a row
index. Here the index is the time in days.

In [ ]:
week = pd.Series(temps, index=days)
ax = week.plot(figsize=(8, 3))
ax.set_xlabel('time (days)')
ax.set_ylabel('temperature (degrees Celsius)')
plt.show()

Two panels, which is how you will usually compare things. Here is a tone and
the same tone with a loud hiss added. The tone is the note A, 440 Hz: the
pressure goes up and down 440 times every second. The hiss is random noise.

In [ ]:
rate = 8000                                # values per second
t = np.arange(0, 1, 1 / rate)              # one second, in seconds
tone = np.sin(2 * np.pi * 440 * t)         # the note A, 440 Hz
noise = np.random.default_rng(3).standard_normal(t.size)
noisy = tone + 2.0 * noise                 # a hiss twice as big
n = 80                                     # the first 10 ms

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(t[:n] * 1000, tone[:n])
axes[0].set_ylabel('clean')
axes[1].plot(t[:n] * 1000, noisy[:n], color='tab:red')
axes[1].set_ylabel('under hiss')
axes[1].set_xlabel('time (ms)')
fig.tight_layout()
plt.show()

> **Careful — The tone is still there and you cannot see it**
>
> The lower panel holds exactly the same tone. The hiss is twice as big and the
> eye finds nothing. A plot is one way to look at data. Listening is another.

Listen to both. The ear finds the pitch where the eye could not.

In [ ]:
from IPython.display import Audio

Audio(noisy, rate=rate)

## SciPy: the rest of the stack

`scipy` is a large collection of tools built on NumPy. We use one of them: a
straight line through points. How has the temperature of a city changed over
forty years? Here are forty yearly mean temperatures. They are invented for
the exercise: a slow rise plus a scatter from one year to the next.

In [ ]:
years = np.arange(1985, 2025)                   # forty years
rng = np.random.default_rng(11)
since = years - 1985                            # years since the first one
yearly = 14.0 + 0.03 * since + rng.normal(0, 0.4, years.size)

print(years[:3], np.round(yearly[:3], 1))

`scipy.stats.linregress` finds the straight line that comes closest to the
points. The **slope** is how many degrees the line rises each year and the
**intercept** is its value in the first year.

In [ ]:
from scipy import stats

line = stats.linregress(since, yearly)

print(f"slope      {line.slope:.3f} degrees per year")
print(f"intercept  {line.intercept:.2f} degrees")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(years, yearly, 'o', label='yearly mean (invented)')
ax.plot(years, line.intercept + line.slope * since, label='straight line')
ax.set_xlabel('year')
ax.set_ylabel('mean temperature (degrees Celsius)')
ax.legend()
fig.tight_layout()
plt.show()

The points scatter around a line that climbs a little every year. Over forty
years the line rises by about a degree. The numbers are invented, so this
shows how the tool works and says nothing about any real city.

## Exercises

### 1. Write, read, check

Here is the register of a second class. Write the names to a text file called
`register_b.txt`, in alphabetical order, one per line. Read them back and
check that the list you read is the sorted list you wrote.

In [ ]:
names_b = ["Giulia", "Franco", "Irene", "Hugo", "Elisa"]

# your code here

> **Solution**
>
> Typed out together in the session. Write your own version first — the error you
> make on the way is the part you keep.


### 2. The marks, two ways

Using `marks.csv`: which students have a mean mark above 25? Do it once with
`csv.DictReader` and once with `pandas` and check that the two agree.

**What to look for.** How many are there?

In [ ]:
# your code here

> **Solution**
>
> Typed out together in the session. Write your own version first — the error you
> make on the way is the part you keep.


### 3. A small HDF5 file of your own

Here is one day of temperatures, one reading per hour. They are made up.
Create an HDF5 file with a group called `day1`. Put the temperatures in it as
a dataset. Store the number of readings per hour and the units as attributes.
Read it back in a separate cell and plot the whole day using only what you
read from the file.

This is the bonus: the plot must take its time axis and its labels from the
file and from nothing else.

In [ ]:
hours_day = np.arange(24)
day = np.round(18 + 6 * np.sin(2 * np.pi * (hours_day - 9) / 24), 1)

# your code here

> **Solution**
>
> Typed out together in the session. Write your own version first — the error you
> make on the way is the part you keep.


## Where this leaves you

You can open a file in four formats, read what is in it and plot it with the
axes labelled. You can listen to an array and draw a straight line through
points. That is the floor the rest of the course builds on.

The next session turns the functions you have written into code you can trust,
keep and share.

**Do.** `reference/modulo_5_files_libraries/exercises/`, which covers the same
ground with more examples.